In [44]:
import sqlite3
import pandas as pd
con=sqlite3.connect(r'C:\Users\muhsi\OneDrive\Desktop\project_3.1_ipl\data\raw\ipl.db')

from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import numpy as np


def q(sql):
     return pd.read_sql(sql, con)

In [45]:
stats.binomtest(613, 1187, 0.5).pvalue

np.float64(0.2700389712415702)

In [46]:
#DOES WINNING TOSS HELP YOU WIN THE MATCH?
# ho-The toss winner wins half the time. The 51.64% we saw is luck, and nothing more.
#h1 -The toss winner does not win half the time

q("""SELECT COUNT(*) AS n,
       SUM(CASE WHEN toss_winner = match_winner
               THEN 1 ELSE 0 END) AS k
FROM matches_clean
WHERE result = 'win';""")

,n,k
0,1187,613


In [47]:
p=stats.binomtest(613,1187,0.5).pvalue
ci=proportion_confint(613,1187,method="wilson")
print(p)
print(ci)

#pvalue<significant leve l THEN the h0 is significant
#pvalue<significant leve l THEN the h0 is  not significant


0.2700389712415702
(0.48799203833275506, 0.5447579131499739)


In [48]:
q("""SELECT toss_decision,
       SUM(CASE WHEN toss_winner = match_winner
           THEN 1 ELSE 0 END) AS won,
       SUM(CASE WHEN toss_winner <> match_winner
           THEN 1 ELSE 0 END) AS lost
FROM   matches_clean
WHERE  result = 'win'
GROUP BY toss_decision;""")

,toss_decision,won,lost
0,bat,184,218
1,field,429,356


In [49]:
table = [[184, 218],
         [429, 356]]

chi2, p, dof, expected = stats.chi2_contingency(table)

print(round(chi2, 2))
print(round(p, 4))

8.04
0.0046


In [50]:
#Does the chasing team really have an advantage?
#h0-Both sides win half the time. The 54.51% we saw is luck.
#h1-A chase does not succeed half the time.
q(""" SELECT COUNT(*) AS n, 
SUM(chase_won)AS k 
FROM v_match_totals;""")

,n,k
0,1187,647


In [51]:
w=stats.binomtest(647, 1187,0.5).pvalue 
l=proportion_confint(647,1187, method='wilson')
print(w)
print(l)

#signfiiciant level = 0.005 
#pvalue<signficcant level then the h0 is signficiant 
#pvalue>signficant level THEN the h0 is not signficant

0.002080449027018435
(0.5166432360570298, 0.5732091949340473)


In [52]:
#Does the chase win rate really fall as the target rises?

# h0-The chase win rate is the same whatever the target. 
# The differences between the bands are luck.

#h1- The chase win rate is not the same in every target band.


q("""
SELECT
    CASE
        WHEN first_innings_runs < 140 THEN 'Under 140'
        WHEN first_innings_runs < 160 THEN '140-159'
        WHEN first_innings_runs < 180 THEN '160-179'
        WHEN first_innings_runs < 200 THEN '180-199'
        ELSE '200+'
    END AS band,

    SUM(chase_won) AS won,
    SUM(1 - chase_won) AS lost

FROM v_match_totals

GROUP BY band;
""")

,band,won,lost
0,140-159,171,70
1,160-179,166,146
2,180-199,83,134
3,200+,46,152
4,Under 140,181,38


In [53]:
table = [[181, 38],
         [171, 70],
         [166,146],
         [83,134],
         [46,152]]
chi2, p, dof, expected = stats.chi2_contingency(table)

print(round(chi2, 2))
print((p,))

197.68
(np.float64(1.1854224475712924e-41),)


In [54]:
#Are defended totals actually bigger than chased ones?

#h0=Both kinds of total average the same. The gap we see is luck
#h1=The two averages are not equal.

q("""SELECT first_innings_runs, chase_won
FROM   v_match_totals;""")

,first_innings_runs,chase_won
0,222,0
1,207,0
2,184,1
3,183,1
4,163,1
...,...,...
1182,75,1
1183,222,1
1184,243,1
1185,155,1


In [55]:
df=pd.read_sql("""SELECT first_innings_runs, chase_won
FROM   v_match_totals;""",con)
defended = df[df.chase_won == 0].first_innings_runs
chased   = df[df.chase_won == 1].first_innings_runs
t, p = stats.ttest_ind(defended, chased, equal_var=False)
# move 6 needs the effect size too
pooled = np.sqrt((defended.std()**2 + chased.std()**2) / 2)
d = (defended.mean() - chased.mean()) / pooled
print(d)
print(pooled)
print(t)
print(p)

0.9162898870801979
30.342879534020167
15.730238965353694
1.17272268263676e-50


In [56]:
#Is the powerplay really faster than the middle overs?

#claim :Teams score faster in the first 
# six overs than they do in overs 7 to 15.
#h0= Both phases score at the same rate.
# Thegap we saw is luck.
#h1=The two run rates are not equal.

In [57]:
df=pd.read_sql("""SELECT match_id, innings,
       6.0 * SUM(CASE WHEN phase='Powerplay' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) AS pp,

       6.0 * SUM(CASE WHEN phase='Middle' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) AS mid

FROM v_ball

WHERE innings IN (1,2)

GROUP BY match_id, innings

HAVING
    SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) > 0
    AND
    SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) > 0;""",con)

In [58]:
t,p=stats.ttest_rel(df.pp, df.mid)
print(round(t,2))
print(round(p,4))
print(p<0.05)

1.53
0.1259
False


In [59]:
#Do left-handers really hit harder than right-handers?
#claim:Left-handed batters have a higher strike rate than right-handed batters.
#h0=Both hit at the same rate. The gap found is luck.
# h1=The two average strike rates are not equal.

df=pd.read_sql("""
SELECT batter, batsman_type,
    100.0*SUM(batter_runs)/SUM(is_legal) AS sr,
    SUM(is_legal) AS balls
FROM v_ball
WHERE batsman_type IS NOT NULL
GROUP BY batter, batsman_type;""",con)
df

,batter,batsman_type,sr,balls
0,A Ashish Reddy,Right hand Bat,145.077720,193
1,A Badoni,Right hand Bat,139.263804,815
2,A Chandila,Right hand Bat,57.142857,7
3,A Chopra,Right hand Bat,74.647887,71
4,A Choudhary,Right hand Bat,125.000000,20
...,...,...,...,...
720,Yashpal Singh,Right hand Bat,72.307692,65
721,Younis Khan,Right hand Bat,42.857143,7
722,Yudhvir Singh,Right hand Bat,137.500000,16
723,Yuvraj Singh,Left hand Bat,130.146711,2113


In [60]:
left= df[df.batsman_type == 'Left hand Bat']
right= df[df.batsman_type == 'Right hand Bat']
t,p=stats.ttest_ind(left.sr, right.sr, equal_var=False)
print(t)
print(p)
print(p<0.05)

2.3574089065088315
0.018993053379198307
True


In [61]:
#Are spinners genuinely cheaper than fast bowlers?
# CLAIM:Spin bowlers concede fewer runs per over than fast bowlers
# h0=Both styles concede the same. The gap found is luck.
# h1=The two average economy rates are not equal

df=pd.read_sql("""
SELECT 
    bowler,

    CASE
        WHEN bowler_type LIKE '%rightarm%'
          OR bowler_type LIKE '%leftarm%'
          OR bowler_type LIKE '%orthodox%'
          OR bowler_type LIKE '%googly%'
          OR bowler_type LIKE '%wrist spin%'
        THEN 'Spin'
        ELSE 'Pace'
    END AS style,

    SUM(bowler_runs) / NULLIF(SUM(is_legal), 0) AS economy,

    SUM(is_legal) AS balls

FROM v_ball
where TRIM(COALESCE(bowler_type, '')) <> ''
GROUP BY bowler, style
HAVING balls >=500;
""",con)
df

,bowler,style,economy,balls
0,A Kumble,Spin,1,965
1,A Mishra,Pace,1,3371
2,A Nehra,Pace,1,1908
3,A Nortje,Pace,1,1120
4,A Symonds,Pace,1,527
...,...,...,...,...
144,YK Pathan,Pace,1,1147
145,YS Chahal,Spin,1,3941
146,Yash Dayal,Pace,1,871
147,Yuvraj Singh,Spin,1,869


In [62]:
df=pd.read_sql("""
SELECT bowler,
       CASE WHEN bowler_type LIKE '%Legbreak%'
              OR bowler_type LIKE '%Offbreak%'
              OR bowler_type LIKE '%Orthodox%'
              OR bowler_type LIKE '%Googly%'
              OR bowler_type LIKE '%Wrist spin%'
            THEN 'Spin' ELSE 'Pace' END AS style,
       6.0*SUM(bowler_runs)/SUM(is_legal) AS economy,
       SUM(is_legal) AS balls
FROM   v_ball
WHERE  TRIM(COALESCE(bowler_type,'')) <> ''
GROUP  BY bowler, style
HAVING balls >= 500;
""",con)

In [63]:
spin= df[df['style'] == 'Spin']
pace= df[df['style'] == 'Pace']
t,p=stats.ttest_ind(spin.economy, pace.economy, equal_var=False)
pooled = np.sqrt((spin.economy.std()**2 + pace.economy.std()**2) / 2)
d = (pace.economy.mean() - spin.economy.mean()) / pooled
print(t)
print(p)
print(p<0.05)
print(d)

-7.075597311698715
1.1249651433007181e-10
True
1.1866831126238373


In [64]:
# Does Sawai Mansingh really favour the chasing team?
# Chasing works better at Sawai Mansingh Stadium than it does elsewhere.

# h0= Chasing at Sawai Mansingh works exactly as well as chasing anywhere else— 54.51% 
# — and the 65.15% we saw is luck.
# h1=The chase rate at this ground is not 54.51%.

df=pd.read_sql("""
    select count(*) as n,
        sum(chase_won) as k
    from v_match_totals
    where venue_clean = 'Sawai Mansingh Stadium';
    """, con)
df

,n,k
0,66,43


In [65]:
p=stats.binomtest(43, 66, 0.5451).pvalue
ci=proportion_confint(43, 66, method='wilson')
print(p)
print(ci)
print(p<0.05)

0.08493446917661332
(0.5311216021303434, 0.7552412593807046)
False


In [66]:
#Has scoring genuinely risen across the seasons?
#Claim:Average innings scores have risen over the nineteen seasons.
#h0-Average scores wander up and down with no trend, and the climb we saw is luck.
#h1-The season and the average score are related.

df=pd.read_sql("""
    select m.season_year,avg(i.runs) as avg_score
    from matches_clean m
    join v_innings i on i.match_id=m.match_id
    group by m.season_year;""", con)
df

,season_year,avg_score
0,2008,154.629310
1,2009,143.157895
2,2010,157.200000
3,2011,145.889655
4,2012,151.709459
5,2013,148.296053
6,2014,157.575000
7,2015,156.683761
8,2016,157.183333
9,2017,159.059322


In [67]:
r, p = stats.pearsonr(df.season_year, df.avg_score)
r_squared = r**2
print(round(r, 3))
print(p)
print(round(r_squared, 3))
print(p < 0.05)

0.862
2.079722060152857e-06
0.743
True


In [68]:
# Is Mumbai's head-to-head record over Chennai real?
# Claim :Mumbai Indians have the better of Chennai Super Kings.
# h0-The two teams are evenly matched, and the 21-19 record is luck.
# h1-Mumbai do not win half of the meetings.

df=pd.read_sql("""SELECT COUNT(*) AS n,
        SUM(CASE WHEN match_winner = 'Mumbai Indians'
            THEN 1 ELSE 0 END) AS k
FROM  matches_clean
WHERE result = 'win'
  AND ((team1 = 'Mumbai Indians'
        AND team2 = 'Chennai Super Kings')
   OR (team1 = 'Chennai Super Kings'
        AND team2 = 'Mumbai Indians'));""",con)

df

,n,k
0,40,21


In [69]:
# Does the chasing advantage change from season to season?
# claim:Some seasons favour the chasing team much more than others.
# h0-The chase win rate is the same in every season, and the differences are luck.
# h1=The chase win rate is not the same in every season.

df=pd.read_sql("""SELECT season_year,
         SUM(chase_won) AS won,
         SUM(1 - chase_won) AS lost
         from v_match_totals 
         group by season_year;""",con)
df

,season_year,won,lost
0,2008,36,22
1,2009,30,26
2,2010,28,31
3,2011,40,32
4,2012,40,34
5,2013,37,37
6,2014,37,22
7,2015,24,32
8,2016,41,19
9,2017,32,26


In [70]:
chi2, p, dof, expected = stats.chi2_contingency(df[['won', 'lost']])
print(round(chi2,2))
print(round(p,4))
print(p<0.05)

20.34
0.3141
False


In [71]:
Do the bowling styles really differ in how much they concede?
# claim:The eight main bowling styles do not all concede runs at the same rate.
# ho-Every style has the same true economy rate,
# and the differences between them are random luck.
# h1-At least one style concedes at a different rate from the rest.

df=pd.read_sql("""SELECT bowler, bowler_type,
       6.0*SUM(bowler_runs)/SUM(is_legal) AS economy,
         SUM(is_legal) AS balls
FROM v_ball
WHERE trim(COALESCE(bowler_type,'')) <> ''
GROUP BY bowler, bowler_type    
having balls >= 500;""",con)
df 

Object `concede` not found.


,bowler,bowler_type,economy,balls
0,A Kumble,Legbreak Googly,6.578238,965
1,A Mishra,Legbreak,7.377633,3371
2,A Nehra,Left arm Medium fast,7.845912,1908
3,A Nortje,Right arm Fast,9.085714,1120
4,A Symonds,"Right arm Medium, Right arm Offbreak",7.673624,527
...,...,...,...,...
144,YK Pathan,Right arm Offbreak,7.401918,1147
145,YS Chahal,Legbreak Googly,8.027912,3941
146,Yash Dayal,Left arm Medium fast,9.575201,871
147,Yuvraj Singh,Slow Left arm Orthodox,7.436133,869


In [72]:
styles = df['bowler_type'].value_counts()
styles = styles[styles >= 8].index
groups = [df[df['bowler_type'] == s].economy for s in styles]
f ,p = stats.f_oneway(*groups)
print(round(f,2))
print(p)
print(len(groups))
print(p<0.05)

6.99
5.916709274557887e-07
8
True
